# 🛸 Fine-Tuning Llama 3 8B Instruct with LoRA for Mermaid UML Generation

This notebook provides the complete setup for fine-tuning Llama 3 (8B Instruct) to generate syntactically correct **Mermaid.js** UML diagrams from software requirements. 

It uses QLoRA (4-bit quantization + LoRA) to train efficiently on consumer-grade GPUs or Google Colab (T4 GPU).

### 1. Install Dependencies
Install the required libraries for quantization, PEFT training, and Weights & Biases logging.

In [1]:
!pip install -q torch transformers datasets peft trl accelerate bitsandbytes wandb sentencepiece


[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


### 2. Imports and Configuration
Define directories and model ID. Make sure you have downloaded or exported your `llama3_uml_dataset.jsonl` from the NestJS backend.

In [ ]:
import os
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer

MODEL_ID = "meta-llama/Meta-Llama-3-8B-Instruct"
DATASET_PATH = "llama3_uml_dataset.jsonl"  # Upload your exported file here
OUTPUT_DIR = "./llama3-mermaid-uml"

if not os.path.exists(DATASET_PATH):
    print("WARNING: Dataset not found. Please upload llama3_uml_dataset.jsonl to the current directory.")

### 3. Load Model in 4-bit (QLoRA)
Requires access to `meta-llama/Meta-Llama-3-8B-Instruct` on Hugging Face (accept the Meta license and login using `huggingface-cli login` if running remotely).

In [ ]:
# Setup 4-bit configuration
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

print("Loading model...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto"
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# Prepare model for quantization-aware training
model = prepare_model_for_kbit_training(model)

### 4. Setup LoRA Parameters
Configure parameter adapters (PEFT) targeting attention weights and projection layers.

In [ ]:
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

### 5. Format Dataset and Split (Train/Eval)

In [ ]:
raw_dataset = load_dataset("json", data_files=DATASET_PATH, split="train")

# Train/Test Split (90% train, 10% test)
dataset_split = raw_dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = dataset_split["train"]
eval_dataset = dataset_split["test"]
print(f"Loaded {len(train_dataset)} train entries and {len(eval_dataset)} evaluation entries.")

def format_prompts(examples):
    texts = []
    for instruction, user_input, output in zip(examples['instruction'], examples['input'], examples['output']):
        text = (
            f"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\n"
            f"{instruction}<|eot_id|>"
            f"<|start_header_id|>user<|end_header_id|>\n\n"
            f"{user_input}<|eot_id|>"
            f"<|start_header_id|>assistant<|end_header_id|>\n\n"
            f"{output}<|eot_id|>"
        )
        texts.append(text)
    return {"text": texts}

formatted_train = train_dataset.map(format_prompts, batched=True)
formatted_eval = eval_dataset.map(format_prompts, batched=True)

### 6. Run SFT Trainer
Set epochs and evaluate at the end of each epoch. Use Cosine schedule and early stopping to find the best checkpoint.

In [ ]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    logging_steps=5,
    num_train_epochs=5,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    bf16=True if torch.cuda.is_bf16_supported() else False,
    optim="paged_adamw_8bit",
    report_to="none"
)

trainer = SFTTrainer(
    model=model,
    train_dataset=formatted_train,
    eval_dataset=formatted_eval,
    peft_config=peft_config,
    dataset_text_field="text",
    max_seq_length=512,
    tokenizer=tokenizer,
    args=training_args,
    data_collator=DataCollatorForSeq2Seq(tokenizer, pad_to_multiple_of=8, return_tensors="pt", padding=True),
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

print("Training model...")
trainer.train()

### 7. Save Fine-Tuned Model

In [ ]:
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Model adapter saved successfully in: {OUTPUT_DIR}")

### 8. Inference / Test Model

In [ ]:
prompt = (
    f"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\n"
    f"Generate Mermaid UML diagram of type CLASS<|eot_id|>"
    f"<|start_header_id|>user<|end_header_id|>\n\n"
    f"Un utilisateur s'authentifie pour commander un produit. Chaque commande a plusieurs articles.<|eot_id|>"
    f"<|start_header_id|>assistant<|end_header_id|>\n\n"
)

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=256)
    
print(tokenizer.decode(outputs[0], skip_special_tokens=True))